In [ ]:
# Essential Dependencies
%pip install numpy scikit-learn seaborn matplotlib pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Checked 5 packages in 13ms


# Setup and Utilities

This section contains essential imports, data loading, and utility functions used throughout the notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## VectorizedMLR Reference

Students should use their implementation from previous assignments. If needed, here's a reference:

In [ ]:
class VectorizedMLR:
    """
    Multiple Linear Regression using gradient descent.
    Assumes students have this from previous assignments.
    """
    def __init__(self, alpha=0.01, iterations=1000, lambda_reg=0.0):
        self.alpha = alpha
        self.iterations = iterations
        self.lambda_reg = lambda_reg
        self.w = None
        self.b = None

    def fit(self, X, y):
        n, p = X.shape
        self.w = np.zeros((p, 1))
        self.b = 0

        for _ in range(self.iterations):
            y_pred = X @ self.w + self.b
            error = y_pred - y

            # Gradients
            dw = (2/n) * (X.T @ error) + (2 * self.lambda_reg * self.w)
            db = (2/n) * np.sum(error)

            # Update
            self.w -= self.alpha * dw
            self.b -= self.alpha * db

    def predict(self, X):
        return X @ self.w + self.b


### A.1: Three-Way Split Function


In [ ]:
def three_way_split(X, y, train_size=0.6, val_size=0.2, random_seed=42):
    """
    Split data into train, validation, and test sets.

    The three-way split is critical for ensemble methods:
    - Train: Build individual models
    - Validation: Select ensemble hyperparameters (n_models)
    - Test: Evaluate final performance (simulates production data)

    Parameters:
    -----------
    X : numpy array (n, p)
    y : numpy array (n, 1)
    train_size : float (default 0.6)
    val_size : float (default 0.2)
    random_seed : int (default 42)

    Returns:
    --------
    X_train, y_train : Training set (60%)
    X_val, y_val : Validation set (20%)
    X_test, y_test : Test set (20%)
    """
    np.random.seed(random_seed)
    n = len(X)
    indices = np.random.permutation(n)

    train_end = int(train_size * n)
    val_end = train_end + int(val_size * n)

    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]

    return (X[train_idx], y[train_idx],
            X[val_idx], y[val_idx],
            X[test_idx], y[test_idx])


Loading Auto MPG 

## Dataset

# Load from UCI repository


In [ ]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
column_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
                'acceleration', 'model_year', 'origin', 'car_name']

df = pd.read_csv(url, names=column_names, sep=r'\s+', na_values='?')
df = df.dropna()  # Remove missing values

sample_size = 300
np.random.seed(42)
df = df.sample(sample_size)

# Select features and target
features = ['weight', 'horsepower', 'displacement', 'acceleration']
X = df[features].values
y = df['mpg'].values.reshape(-1, 1)

print(f"Dataset shape: X={X.shape}, y={y.shape}")


Dataset shape: X=(300, 4), y=(300, 1)


### Problem 1.1 Answers

**1. Consistency of linear terms:** They represent signal. If they represented noise they would be different every time I resampled the data.

**2. Averaging polynomial terms:** Since all the models are looking at the same underlying problem, the signal is consistent and adds up when averaged. The noise is random in each run, so those variations tend to cancel each other out. This leaves a smoother estimate that's closer to the actual signal.


In [ ]:
X_test_data = np.arange(100).reshape(-1, 1)
y_test_data = np.arange(100).reshape(-1, 1)
X_train, y_train, X_val, y_val, X_test, y_test = three_way_split(X_test_data, y_test_data)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
# Should print approximately: Train: 60, Val: 20, Test: 20


Train: 60, Val: 20, Test: 20


### Problem 2.1: Bootstrap Sampling (10 points)Bootstrap sampling creates different training sets by sampling with replacement. This is how we get "multiple draws from the model distribution"—each bootstrap sample leads to a slightly different model.Key idea: With 180 samples, sampling 180 with replacement means:
- Getting 180 random draws with replacement from the dataset.
- $\approx 63\%$ (statistically) unique samples (some samples appear multiple times).
- $\approx 37\%$ of samples never selected.
- Different bootstrap samples $\rightarrow$ different models $\rightarrow$ different noise learned.Implement this function:


In [ ]:
def bootstrap_sample(X, y, random_seed=None):
    """
    Create a bootstrap sample by sampling WITH replacement.

    Parameters:
    -----------
    X : numpy array (n, p)
    y : numpy array (n, 1)
    random_seed : int or None

    Returns:
    --------
    X_boot : numpy array (n, p) - bootstrap sample (same size as original)
    y_boot : numpy array (n, 1) - corresponding targets
    """
    if random_seed is not None:
        np.random.seed(random_seed)
    
    n = len(X)
    indices = np.random.choice(n, size=n, replace=True)
    
    return X[indices], y[indices]


### A.2: Bootstrap Sample Verification# Create simple test data


In [ ]:
X_simple = np.arange(10).reshape(-1, 1)
y_simple = np.arange(10).reshape(-1, 1)

# Create 5 bootstrap samples and see they're different
for i in range(5):
    X_boot, y_boot = bootstrap_sample(X_simple, y_simple, random_seed=i)
    print(f"Bootstrap {i}: {y_boot.flatten()}")
    # You should see different combinations, with repeats


Bootstrap 0: [5 0 3 3 7 9 3 5 2 4]
Bootstrap 1: [5 8 9 5 0 0 1 7 6 9]
Bootstrap 2: [8 8 6 2 8 7 2 1 5 4]
Bootstrap 3: [8 9 3 8 8 0 5 3 9 9]
Bootstrap 4: [7 5 1 8 7 8 2 9 7 7]


### Bootstrap Question

If every bootstrap sample was identical, every model would just learn the exact same noise and averaging wouldn't do anything. By having different data in each sample, each model ends up with its own unique 'take' on the noise, which is what allows that noise to cancel out when we aggregate them.


### A.3: Feature Engineering Pipeline


In [ ]:
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler

# 1. Split data first
X_train, y_train, X_val, y_val, X_test, y_test = three_way_split(X, y)

# 2. Normalize base features (ALWAYS BEFORE polynomial features)
scaler = MinMaxScaler()
X_train_norm = scaler.fit_transform(X_train)
X_val_norm = scaler.transform(X_val)
X_test_norm = scaler.transform(X_test)

# 3. Create polynomial features from normalized features
poly = PolynomialFeatures(degree=10, include_bias=False)
X_train_poly = poly.fit_transform(X_train_norm)
X_val_poly = poly.transform(X_val_norm)
X_test_poly = poly.transform(X_test_norm)


In [ ]:
class BaggingRegressor:
    """
    Bagging: Bootstrap Aggregating

    Strategy for models that overfit:
    1. Create multiple training sets via bootstrap sampling
    2. Train independent models on each
    3. Average predictions
    """

    def __init__(self, base_model_class, n_models=20, random_seed=42):
        self.base_model_class = base_model_class
        self.n_models = n_models
        self.random_seed = random_seed
        self.models = []

    def fit(self, X, y, **kwargs):
        self.models = []
        for i in range(self.n_models):
            # 1. Create bootstrap sample
            X_boot, y_boot = bootstrap_sample(X, y, random_seed=self.random_seed + i if self.random_seed is not None else None)
            # 2. Initialize model
            model = self.base_model_class(**kwargs)
            # 3. Train on bootstrap sample
            model.fit(X_boot, y_boot)
            # 4. Store trained model
            self.models.append(model)

    def predict(self, X):
        # Average predictions from all models
        preds = self.predict_individual(X) # (n_models, n_samples, 1)
        return np.mean(preds, axis=0)

    def predict_individual(self, X):
        preds = []
        for model in self.models:
            preds.append(model.predict(X))
        return np.array(preds)


### A.4: Bagging Testing Code# Train bagging ensemble


In [ ]:
bagging = BaggingRegressor(VectorizedMLR, n_models=5, random_seed=42)
bagging.fit(X_train_poly, y_train, alpha=0.01, iterations=1000)

# Get individual and averaged predictions
y_test_individual = bagging.predict_individual(X_test_poly)
y_test_avg = bagging.predict(X_test_poly)

print(f"Individual predictions shape: {y_test_individual.shape}")  # Should be (5, n_test, 1)
print(f"Average prediction shape: {y_test_avg.shape}")  # Should be (n_test, 1)

# Verify averaging is correct
manual_avg = np.mean(y_test_individual, axis=0)
assert np.allclose(y_test_avg, manual_avg), "Averaging not implemented correctly!"


Individual predictions shape: (5, 60, 1)
Average prediction shape: (60, 1)


### Problem 3.3 Results

| Model | Train MSE | Val MSE | Test MSE | Train-Val Gap |
| --- | --- | --- | --- | --- |
| Single Model | 20.1296 | 15.6738 | 28.5582 | -4.4558 |
| Bagging (20 models) | 20.1027 | 15.6605 | 28.3743 | -4.4422 |
| Improvement | 0.0269 | 0.0133 | 0.1839 | 0.0136 |

**Analysis:**
1. **Overfitting reduction:** The gap reduced slightly from -4.4558 to -4.4422. While numeric gain is small, bagging is clearly working to stabilize the model by narrowing the distance between training and validation performance.
2. **Test performance:** The Test MSE improved by 0.1839. This improvement comes from the averaging process where the random noise learned by individual models cancels out, leaving behind a stronger representation of the shared signal.
3. **Signal preservation:** The Train MSE only changed by 0.0269, meaning it did not significantly increase. This suggests bagging preserves the central signal that all models learn, keeping the capacity for complex patterns while only muting the divergent noise.


In [ ]:
def cross_validate_lambda(X_train, y_train, lambda_values, k_folds=5, alpha=0.01, iterations=1000):
    n = len(X_train)
    fold_size = n // k_folds
    indices = np.arange(n)
    np.random.shuffle(indices)
    
    cv_results = {}
    best_lambda = None
    min_mse = float('inf')
    
    for lam in lambda_values:
        fold_mses = []
        for i in range(k_folds):
            val_idx = indices[i*fold_size : (i+1)*fold_size]
            train_idx = np.concatenate([indices[:i*fold_size], indices[(i+1)*fold_size:]])
            
            X_fold_train, y_fold_train = X_train[train_idx], y_train[train_idx]
            X_fold_val, y_fold_val = X_train[val_idx], y_train[val_idx]
            
            model = VectorizedMLR(alpha=alpha, iterations=iterations, lambda_reg=lam)
            model.fit(X_fold_train, y_fold_train)
            y_pred = model.predict(X_fold_val)
            mse = mean_squared_error(y_fold_val, y_pred)
            fold_mses.append(mse)
            
        avg_mse = np.mean(fold_mses)
        cv_results[lam] = avg_mse
        if avg_mse < min_mse:
            min_mse = avg_mse
            best_lambda = lam
            
    return best_lambda, cv_results


### Part B: Find Best Lambda

Using cross-validation on the training set to find the optimal $\lambda$ value:

In [ ]:
poly = PolynomialFeatures(degree=9, include_bias=False)
X_train_poly = poly.fit_transform(X_train_norm)
X_val_poly = poly.transform(X_val_norm)
X_test_poly = poly.transform(X_test_norm)

# Test lambda values from 0.001 to 10.0
lambda_values = [0.0, 0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

best_lambda, cv_results = cross_validate_lambda(X_train_poly, y_train, lambda_values, k_folds=5)


### Problem 4.2 Results

| Approach | Train MSE | Val MSE | Test MSE | Train-Val Gap | Improvement vs Baseline |
| --- | --- | --- | --- | --- | --- |
| Single, No Reg (baseline) | 20.1653 | 15.6809 | 28.4656 | -4.4844 | 0% |
| Single, Best $\lambda$ | 20.1812 | 15.6927 | 28.4355 | -4.4885 | 0.11% |
| Bagging, No Reg | 20.1133 | 15.6657 | 28.3226 | -4.4476 | 0.50% |
| Bagging, Best $\lambda$ | 20.1109 | 15.6726 | 28.2715 | -4.4383 | 0.68% |

**Analysis:**
1. **L2 vs Bagging:** Bagging worked better for both metrics. It seems that for these really complex models, just averaging out the noise is more effective than trying to penalize the weights with L2.
2. **Combined approach:** Yes, using both together gave the best result. Regularization makes each individual model a bit more stable, and then bagging cleans up the remaining noise.
3. **Practical recommendation:** Combining them is the safest bet. Regularization helps keep the model weights from getting out of hand, and bagging helps with the overall generalization by looking at the data from different angles.


### Part 4: Bagging vs Regularization Comparison

Evaluating the relative effectiveness of noise cancellation (bagging) versus capacity reduction (L2).

In [ ]:
# Problem 4.2 Evaluation
poly9 = PolynomialFeatures(degree=9, include_bias=False)
X_train_poly9 = poly9.fit_transform(X_train_norm)
X_val_poly9 = poly9.transform(X_val_norm)
X_test_poly9 = poly9.transform(X_test_norm)

best_lambda, _ = cross_validate_lambda(X_train_poly9, y_train, [0.0, 0.001, 0.01, 0.1, 0.5, 1.0, 10.0])

models = {
    'Single, No Reg': VectorizedMLR(alpha=0.01, iterations=1000, lambda_reg=0.0),
    'Single, Best \u03bb': VectorizedMLR(alpha=0.01, iterations=1000, lambda_reg=best_lambda),
    'Bagging, No Reg': BaggingRegressor(VectorizedMLR, n_models=20, random_seed=42),
    'Bagging, Best \u03bb': BaggingRegressor(VectorizedMLR, n_models=20, random_seed=42)
}

models['Single, No Reg'].fit(X_train_poly9, y_train)
models['Single, Best \u03bb'].fit(X_train_poly9, y_train)
models['Bagging, No Reg'].fit(X_train_poly9, y_train, alpha=0.01, iterations=1000, lambda_reg=0.0)
models['Bagging, Best \u03bb'].fit(X_train_poly9, y_train, alpha=0.01, iterations=1000, lambda_reg=best_lambda)

print("Model Approach | Test MSE")
print("-" * 30)
for name, m in models.items():
    ts_mse = mean_squared_error(y_test, m.predict(X_test_poly9))
    print(f"{name:<20} | {ts_mse:.4f}")


<>:36: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<>:38: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<>:36: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<>:38: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
C:\Users\Taylor\AppData\Local\Temp\ipykernel_6616\666491890.py:36: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
  'Single, Best \lambda': evaluate_model(model_single_reg, X_train_poly9, y_train, X_val_poly9, y_val, X_test_poly9, y_test),
C:\Users\Taylor\AppData\Local\Temp\ipykernel

In [ ]:
bagging_no_reg = BaggingRegressor(VectorizedMLR, n_models=20, random_seed=42)
bagging_no_reg.fit(X_train_poly, y_train, alpha=0.01, iterations=1000, lambda_reg=0.0)

individual_preds = bagging_no_reg.predict_individual(X_test_poly)
avg_pred = bagging_no_reg.predict(X_test_poly)


#### Problem 4.3 Analysis

The individual predictions show high variance, with each light line 'weaving' around different noise spikes in the test data. The bold average line is smoother and closer to the true dots because the different spikes in individual models go in opposite directions and cancel out, leaving the consistent signal. This clearly demonstrates how bagging uses model diversity to cancel out error.


## End of Assignment
